In [26]:
import torch

from tqdm import tqdm
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [28]:
transform = transforms.Compose([
    transforms.ToTensor(),           # (1, 28, 28)
    #transforms.Flatten(),            # (784)  
    transforms.Normalize((0.1307,), (0.3081,))
])

train_data = datasets.MNIST("./data", train=True, download=True, transform=transform)
test_data = datasets.MNIST("./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=100, shuffle=False)

In [34]:
sample_data, sample_label = train_data[0]
sample_label

5

In [ ]:
class NnetMNISTclassificator(nn.Module):
    def __init__(self, inputs, outputs):
        super().__init__()
        
        self.flatten = nn.Flatten()
        self.input_layer = nn.Linear(inputs, outputs)
        self.activation_function = nn.functional.relu

    def forward(self, x):
        x = self.flatten(x)
        x = self.input_layer(x)
        x = self.activation_function(x)
        return x


In [42]:
model = NnetMNISTclassificator(784, 10)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

epochs = 5

for epoch in tqdm(range(epochs)):
    model.train()
    for index, data in tqdm(enumerate(train_loader)):
        x_train, y_train = data
        
        optimizer.zero_grad()
        y_pred = model(x_train)
        
        loss = loss_function(y_pred, y_train)
        loss.backward()

        optimizer.step()



938it [00:11, 80.26it/s]:00<?, ?it/s]
938it [00:11, 79.81it/s]:11<00:46, 11.69s/it]
938it [00:12, 75.33it/s]:23<00:35, 11.73s/it]
938it [00:12, 77.30it/s]:35<00:24, 12.06s/it]
938it [00:11, 79.49it/s]:48<00:12, 12.09s/it]
100%|██████████| 5/5 [00:59<00:00, 11.97s/it]


In [43]:
correct = 0
total = 0

model.eval()
with torch.no_grad():
    for data in tqdm(test_loader):
        x_test, y_test = data
        outputs = model(x_test)
        _, predicted = torch.max(outputs.data, 1)

        total += y_test.size(0)
        correct += (predicted == y_test).sum().item()

print(f"Accuracy: {correct / total}")

100%|██████████| 100/100 [00:01<00:00, 55.80it/s]

Accuracy: 0.8161
